In [1]:
import pandas as pd

In [19]:
df=pd.read_csv("ab_data.csv")

In [20]:
print(df.shape)
print(df.columns.tolist())

(294478, 5)
['user_id', 'timestamp', 'group', 'landing_page', 'converted']


In [21]:
df.head()


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [22]:
df.shape

(294478, 5)

In [23]:
df.isnull().sum()

,0
user_id,0
timestamp,0
group,0
landing_page,0
converted,0


In [24]:
valid_df = df[
    ((df["group"] == "control") & (df["landing_page"] == "old_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
]

In [25]:
valid_df.shape

(290585, 5)

In [28]:
valid_df["group"].value_counts()

,count
group,
treatment,145311
control,145274


In [29]:
valid_df.groupby("group")["converted"].sum()

,converted
group,
control,17489
treatment,17264


In [30]:
comparison = valid_df.groupby("group").agg(
    Total_Users=("user_id", "count"),
    Conversions=("converted", "sum")
)

comparison["Non_Conversions"] = (
    comparison["Total_Users"] - comparison["Conversions"]
)

comparison["Conversion_Rate"] = (
    comparison["Conversions"] / comparison["Total_Users"] * 100
)

comparison

,Total_Users,Conversions,Non_Conversions,Conversion_Rate
group,,,,
control,145274,17489,127785,12.038630
treatment,145311,17264,128047,11.880725


In [31]:
control_rate = comparison.loc["control", "Conversion_Rate"]
treatment_rate = comparison.loc["treatment", "Conversion_Rate"]

absolute_difference = treatment_rate - control_rate
relative_lift = (absolute_difference / control_rate) * 100

print("Control Conversion Rate:", round(control_rate, 2), "%")
print("Treatment Conversion Rate:", round(treatment_rate, 2), "%")
print("Absolute Difference:", round(absolute_difference, 2), "percentage points")
print("Relative Lift:", round(relative_lift, 2), "%")

Control Conversion Rate: 12.04 %
Treatment Conversion Rate: 11.88 %
Absolute Difference: -0.16 percentage points
Relative Lift: -1.31 %


In [32]:
from statsmodels.stats.proportion import proportions_ztest

# Control group
control_conversions = 17489
control_users = 145274

# Treatment group
treatment_conversions = 17264
treatment_users = 145311

# Conversions and total users
conversions = [control_conversions, treatment_conversions]
users = [control_users, treatment_users]

# Two-proportion z-test
z_stat, p_value = proportions_ztest(conversions, users)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: 1.3116075339133115
P-value: 0.18965258971881804


In [33]:
import math

control_rate = 17489 / 145274
treatment_rate = 17264 / 145311

difference = treatment_rate - control_rate

# Standard error
se = math.sqrt(
    (control_rate * (1 - control_rate) / 145274) +
    (treatment_rate * (1 - treatment_rate) / 145311)
)

# 95% Confidence Interval
margin_of_error = 1.96 * se

lower = difference - margin_of_error
upper = difference + margin_of_error

print("Difference:", round(difference * 100, 2), "percentage points")
print("95% CI Lower:", round(lower * 100, 2), "percentage points")
print("95% CI Upper:", round(upper * 100, 2), "percentage points")

Difference: -0.16 percentage points
95% CI Lower: -0.39 percentage points
95% CI Upper: 0.08 percentage points


In [34]:
# Effect size

absolute_effect = treatment_rate - control_rate
relative_effect = (absolute_effect / control_rate) * 100

print("Absolute Effect:", round(absolute_effect * 100, 2), "percentage points")
print("Relative Effect:", round(relative_effect, 2), "%")

Absolute Effect: -0.16 percentage points
Relative Effect: -1.31 %
